# MiniMax H3 — Kaggle GPU T4×2 自动视频生成
此 Notebook 从 GitHub 仓库同步三个用户提供的 API 工作流，保留 T2V、I2V、R2V 模式。默认只运行 T2V 无参考图的短测试；I2V/R2V 必须先准备真实图片，不会伪造参考素材。
**双卡说明：** Kaggle 分配 T4×2，但标准 ComfyUI 单视频生成实际使用 GPU0，尚未实现双卡协同计算。


In [ ]:
import os, sys, subprocess
from pathlib import Path
ROOT=Path('/kaggle/working/shipinshencheng')
if (ROOT/'.git').exists():
    subprocess.run(['git','-C',str(ROOT),'pull','--ff-only'],check=True)
else:
    subprocess.run(['git','clone','--depth=1','https://github.com/mjw3177109/shipinshencheng.git',str(ROOT)],check=True)
print('GitHub project synced:', ROOT)


## 选择视频类型与生成参数
T2V 使用你提供的文生视频工作流；I2V 和 R2V 需要把相应参考图片放在 `/kaggle/working/h3_inputs/`，且文件名与原工作流一致。为保证 Kaggle 有机会先跑通，首次测试为 384×224、22 帧、Turbo 8 步（R2V Turbo 4 步）。长期生成应先通过验收，再提高分辨率和时长。
**磁盘配置：** `compact` 默认约 42GB（仅一套工作流模型，包含 Turbo）；`balanced` 原精度约 53.5GB；`experimental-w6a8` 约 37GB，T4 可用性尚未验收。三个配置均受 Kaggle 的磁盘及 RAM 限制，不保证能生成视频。


In [ ]:
import os
MODE='t2v' # 't2v' / 'i2v' / 'r2v'; the current default uses no uploaded images
STORAGE_PROFILE='compact'  # compact / balanced / experimental-w6a8
TURBO=True
WIDTH=384
HEIGHT=384 if MODE=='i2v' else 224
FRAMES=22 # H3 length uses 17*k+5. 124 corresponds to about 5.2 seconds at 24fps
os.environ.update({'H3_MODE':MODE, 'H3_STORAGE_PROFILE':STORAGE_PROFILE,
                   'H3_TURBO':'1' if TURBO else '0',
                   'H3_WIDTH':str(WIDTH),'H3_HEIGHT':str(HEIGHT),'H3_FRAMES':str(FRAMES)})
subprocess.run([sys.executable,'-u',str(ROOT/'kaggle'/'bootstrap_h3.py')],check=True)
subprocess.run([sys.executable,'-u',str(ROOT/'kaggle'/'run_from_sources.py')],check=True)


## 视频结果与验收报告
视频会保存在 `/kaggle/working/h3_videos/`；请从 Kaggle Output 下载。报告为 `render_report.json`，错误日志为 `comfyui.log`。当模型下载、CPU RAM 或显存不足时，不会生成虚假的 MP4。


In [ ]:
import json
from pathlib import Path
from IPython.display import display,Video
OUT=Path('/kaggle/working/h3_videos')
report=OUT/'render_report.json'
if report.exists(): print(report.read_text(encoding='utf-8')[:4000])
videos=sorted(OUT.rglob('*.mp4'),key=lambda p:p.stat().st_mtime,reverse=True)
print('Generated MP4:',len(videos))
if videos: display(Video(str(videos[0]),embed=True,width=640))
